# Day 043 Project: Chat with a SQL Database

## What You're Building

A natural-language query interface for a retail sales database. The user types a question in English; the system generates SQL, validates it, runs it, and returns the result.

**Deliverable:** `ask_db(conn, question)` answers at least 5 natural-language questions correctly, and `_run_project_checks()` passes.

## Project Requirements

1. Set up the database with `setup_db(conn)`
2. Call `get_db_schema(conn)` — inspect what the LLM will see
3. Ask at least 5 questions with `ask_db`; save results in `answers`
4. Verify the guardrail rejects a dangerous SQL string
5. Show the schema + one question + SQL + result in a readable format

In [ ]:
import warnings
warnings.filterwarnings('ignore')


import sqlite3

def setup_db(conn):
    cur = conn.cursor()
    cur.execute('''
        CREATE TABLE IF NOT EXISTS orders (
            order_id  INTEGER PRIMARY KEY,
            product   TEXT,
            category  TEXT,
            region    TEXT,
            price     REAL,
            quantity  INTEGER,
            revenue   REAL
        )''')
    cur.execute('''
        CREATE TABLE IF NOT EXISTS products (
            product    TEXT PRIMARY KEY,
            category   TEXT,
            unit_price REAL
        )''')
    rows = [
        (1,'Widget','Electronics','North',25.0,10,250.0),
        (2,'Gadget','Electronics','South',150.0,3,450.0),
        (3,'Widget','Electronics','South',25.0,5,125.0),
        (4,'Doohickey','Accessories','East',8.0,50,400.0),
        (5,'Gadget','Electronics','East',150.0,7,1050.0),
        (6,'Widget','Electronics','East',25.0,4,100.0),
        (7,'Doohickey','Accessories','North',8.0,20,160.0),
        (8,'Gadget','Electronics','North',150.0,2,300.0),
        (9,'Widget','Electronics','West',25.0,6,150.0),
        (10,'Doohickey','Accessories','South',8.0,15,120.0),
        (11,'Thingamajig','Accessories','North',200.0,1,200.0),
        (12,'Thingamajig','Accessories','East',200.0,4,800.0),
    ]
    cur.executemany(
        'INSERT OR IGNORE INTO orders VALUES (?,?,?,?,?,?,?)', rows
    )
    products = [
        ('Widget','Electronics',25.0),
        ('Gadget','Electronics',150.0),
        ('Doohickey','Accessories',8.0),
        ('Thingamajig','Accessories',200.0),
    ]
    cur.executemany(
        'INSERT OR IGNORE INTO products VALUES (?,?,?)', products
    )
    conn.commit()


def run_query(conn, sql, params=()):
    cur = conn.cursor()
    cur.execute(sql, params)
    cols = [col[0] for col in cur.description]
    return [dict(zip(cols, row)) for row in cur.fetchall()]


def get_db_schema(conn) -> str:
    cur = conn.cursor()
    cur.execute(
        "SELECT name, sql FROM sqlite_master WHERE type='table' ORDER BY name"
    )
    rows = cur.fetchall()
    if not rows:
        return 'No tables found.'
    parts = []
    for name, ddl in rows:
        parts.append(f'Table: {name}')
        parts.append(ddl)
        parts.append('')
    return '\n'.join(parts).strip()


def build_sql_prompt(question: str, schema_str: str) -> str:
    return (
        'You are a SQL expert. Write a SQLite SELECT query to answer the question.\n\n'
        'Requirements:\n'
        '- Use only SELECT statements.\n'
        '- The database schema is provided below.\n'
        '- Respond with ONLY a fenced SQL code block, no explanation.\n\n'
        f'Schema:\n{schema_str}\n\n'
        f'Question: {question}'
    )


import re

def extract_sql(response: str) -> str:
    fence = '`' * 3
    match = re.search(fence + r'sql\s*(.*?)' + fence, response, re.DOTALL)
    if match:
        return match.group(1).strip()
    match = re.search(fence + r'\s*(.*?)' + fence, response, re.DOTALL)
    if match:
        return match.group(1).strip()
    return response.strip()


import re

def is_safe_sql(sql: str) -> bool:
    normalized = re.sub(r'--[^\n]*', '', sql)
    normalized = re.sub(r'/\*.*?\*/', '', normalized, flags=re.DOTALL)
    normalized = normalized.strip().lower()
    if not normalized.startswith('select'):
        return False
    if ';' in normalized:
        return False
    return True


import ollama

def ask_db(conn, question: str, model: str = 'llama3.2') -> str:
    schema = get_db_schema(conn)
    prompt = build_sql_prompt(question, schema)
    resp   = ollama.chat(model=model,
                         messages=[{'role': 'user', 'content': prompt}])
    sql    = extract_sql(resp['message']['content'])
    if not is_safe_sql(sql):
        return f'Unsafe SQL rejected: {sql[:120]}'
    try:
        rows = run_query(conn, sql)
    except Exception as e:
        return f'Query error: {e}'
    if not rows:
        return 'No results found.'
    return str(rows)


conn = sqlite3.connect(':memory:')
setup_db(conn)
print('Database ready.')

## Step 1 — Inspect the Schema

In [ ]:
schema = get_db_schema(conn)
print(schema)

## Step 2 — Ask Natural Language Questions

In [ ]:
questions = [
    'How many orders are there in total?',
    'What is the total revenue from all orders?',
    'Which product had the highest total revenue?',
    'How many orders came from the East region?',
    'What is the average revenue per order in the Electronics category?',
]

answers = []
for q in questions:
    result = ask_db(conn, q)
    answers.append({'question': q, 'answer': result})
    print(f'Q: {q}')
    print(f'A: {result[:120]}')
    print()

## Step 3 — Show the Full Pipeline for One Question

In [ ]:
q = 'Which region had the highest total revenue?'

schema_str = get_db_schema(conn)
prompt = build_sql_prompt(q, schema_str)
import ollama
resp = ollama.chat(model='llama3.2',
                   messages=[{'role': 'user', 'content': prompt}])
sql_raw = resp['message']['content']
sql = extract_sql(sql_raw)

print('Generated SQL:')
print(sql)
print()
print('Safe?', is_safe_sql(sql))
if is_safe_sql(sql):
    try:
        rows = run_query(conn, sql)
        print('Result:', rows)
    except Exception as e:
        print(f'Query error: {e}')

## Project Checks

In [ ]:
def _run_project_checks():
    total = 5
    passed = 0

    # Check 1: schema is non-empty and mentions both tables
    try:
        s = get_db_schema(conn)
        assert isinstance(s, str) and len(s) > 0
        assert 'orders' in s.lower() and 'products' in s.lower()
        passed += 1; print('\u2705 Check 1: get_db_schema returns both table names')
    except Exception as e:
        print(f'\u274c Check 1: {e}')

    # Check 2: is_safe_sql correctly allows SELECT
    try:
        assert is_safe_sql('SELECT * FROM orders') is True
        passed += 1; print('\u2705 Check 2: is_safe_sql allows SELECT')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: is_safe_sql rejects DROP
    try:
        assert is_safe_sql('DROP TABLE orders') is False
        passed += 1; print('\u2705 Check 3: is_safe_sql rejects DROP')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: answers list has at least 5 entries
    try:
        assert 'answers' in globals(), 'answers not defined — run Step 2'
        assert len(answers) >= 5, \
            f'expected >= 5 answers, got {len(answers)}'
        assert all(isinstance(a['answer'], str) for a in answers)
        passed += 1; print(f'\u2705 Check 4: {len(answers)} questions answered')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: ask_db returns a string for a direct count question
    try:
        r = ask_db(conn, 'How many orders are in the database?')
        assert isinstance(r, str) and len(r) > 0
        passed += 1; print(f'\u2705 Check 5: ask_db returns a string ({r[:60]})')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Project complete!')
    print(f'\nScore: {passed}/{total}')


_run_project_checks()

## Bonus Challenges

- Log the generated SQL alongside the answer so you can audit it
- Add a retry loop: if the SQL is unsafe or raises an error, send the error back to the LLM and ask it to fix the query
- Try questions the model struggles with (e.g. 'Show me the running total of revenue ordered by order_id') and analyse the failure
- On Day 44 you will connect to a file-backed SQLite database and use SQLAlchemy — ask_db will work unchanged on any connection object